# 07. Out-of-Sample Test

Este notebook evalúa el rendimiento fuera de muestra (Out-of-Sample / OOS) de los modelos congelados (frozen models) entrenados previamente, garantizando la ausencia de sesgos de anticipación (look-ahead bias) o filtración de datos (data leakage).

El objetivo fundamental es auditar la capacidad predictiva y la consistencia estadística de las señales generadas en un entorno no visto. A través del análisis del Rank Information Coefficient (Rank IC), la estabilidad temporal de la señal, la precisión direccional (Hit Rate) y la monotonía por deciles, se dictaminará si el modelo conserva su robustez cuantitativa antes de proceder a la fase de simulación de carteras y backtesting.


## 1. Imports & Configuration

### 1.1 Librerias

In [1]:
import sys
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import random
import warnings
import joblib

from pathlib import Path

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

In [2]:
# =============================================================================
# Reproducibility
# =============================================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)


# =============================================================================
# Warnings
# =============================================================================

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

## 2. Load Frozen Model & Configuration


### 2.1 Load final model

In [3]:
final_ridge_rank = joblib.load(
    "../data/model_results/final_model/final_ridge_rank.joblib"
)

final_xgb_rank = joblib.load(
    "../data/model_results/final_model/final_xgb_rank.joblib"
)

final_rf_rank = joblib.load(
    "../data/model_results/final_model/final_rf_rank.joblib"
)

print("Final models loaded successfully.")

print(f"Ridge:        {type(final_ridge_rank).__name__}")
print(f"XGBoost:      {type(final_xgb_rank).__name__}")
print(f"Random Forest:{type(final_rf_rank).__name__}")

Final models loaded successfully.
Ridge:        Ridge
XGBoost:      XGBRegressor
Random Forest:RandomForestRegressor


### 2.2 Load model metadata

In [4]:
import json

with open(
    "../data/model_results/final_model/final_model_configurations.json",
    "r",
    encoding="utf-8",
) as f:

    final_model_configurations = json.load(f)

print("Final model configurations loaded successfully.")

Final model configurations loaded successfully.


### 2.3 Verify model configuration

In [5]:
with open(
    "../data/model_results/final_model/training_metadata.json",
    "r",
    encoding="utf-8",
) as f:

    training_metadata = json.load(f)

print("Training metadata loaded successfully.")

Training metadata loaded successfully.


### 2.4 Verify frozen configuration

In [6]:
# -----------------------------------------------------------------------------
# Verify loaded models
# -----------------------------------------------------------------------------

print("\nModels loaded:")

print(
    f"  Ridge:         {type(final_ridge_rank).__name__}"
)

print(
    f"  XGBoost:       {type(final_xgb_rank).__name__}"
)

print(
    f"  Random Forest: {type(final_rf_rank).__name__}"
)


# -----------------------------------------------------------------------------
# Verify model configurations
# -----------------------------------------------------------------------------

print("\nFrozen model configurations:")

for model_name, configuration in final_model_configurations.items():

    print(f"\n{model_name}:")
    
    if isinstance(configuration, dict):

        for parameter, value in configuration.items():

            print(
                f"  {parameter:<25}: {value}"
            )

    else:

        print(
            f"  {configuration}"
        )


# -----------------------------------------------------------------------------
# Verify training metadata
# -----------------------------------------------------------------------------

print("\nTraining metadata:")

if isinstance(training_metadata, dict):

    for section, values in training_metadata.items():

        print(f"\n[{section}]")

        if isinstance(values, dict):

            for key, value in values.items():

                print(
                    f"  {key:<25}: {value}"
                )

        else:

            print(
                f"  {values}"
            )


print("\n" + "=" * 80)
print("FROZEN CONFIGURATION VERIFICATION COMPLETED")
print("=" * 80)


Models loaded:
  Ridge:         Ridge
  XGBoost:       XGBRegressor
  Random Forest: RandomForestRegressor

Frozen model configurations:

XGBoost:
  representation           : Percentile Rank
  features                 : ['momentum_12_1_win_rank', 'upside_volatility_win_rank', 'log10_amihud_win_rank']
  target                   : forward_return_21d
  hyperparameters          : {'learning_rate': 0.05143828405076928, 'max_depth': 4, 'min_child_weight': 9.726261649881026, 'subsample': 0.9100531293444458, 'colsample_bytree': 0.9757995766256756, 'gamma': 4.474136752138244, 'reg_alpha': 0.09761125443110447, 'reg_lambda': 4.869640941520899}
  random_state             : 42

Ridge:
  representation           : Percentile Rank
  features                 : ['momentum_12_1_win_rank', 'upside_volatility_win_rank', 'log10_amihud_win_rank']
  target                   : forward_return_21d
  hyperparameters          : {'alpha': 506.1576888752306}
  random_state             : 42

Random Forest:
  repre

In [7]:
# =============================================================================
# Frozen Experiment Definition
# =============================================================================

EXPECTED_FEATURES = [
    "momentum_12_1_win_rank",
    "upside_volatility_win_rank",
    "log10_amihud_win_rank",
]

EXPECTED_REPRESENTATION = "Percentile Rank"
EXPECTED_TARGET = "forward_return_21d"
EXPECTED_HORIZON = 21
EXPECTED_SEED = 42


print("\n" + "=" * 80)
print("FROZEN EXPERIMENT DEFINITION")
print("=" * 80)

print("\nFeatures:")
for feature in EXPECTED_FEATURES:
    print(f"  - {feature}")

print(f"\nRepresentation: {EXPECTED_REPRESENTATION}")
print(f"Target:         {EXPECTED_TARGET}")
print(f"Horizon:        {EXPECTED_HORIZON} trading days")
print(f"Seed:            {EXPECTED_SEED}")

print("\n" + "=" * 80)


FROZEN EXPERIMENT DEFINITION

Features:
  - momentum_12_1_win_rank
  - upside_volatility_win_rank
  - log10_amihud_win_rank

Representation: Percentile Rank
Target:         forward_return_21d
Horizon:        21 trading days
Seed:            42



## 3. Load OOS Dataset

El período Out-of-Sample se define de forma estrictamente posterior al conjunto utilizado durante el desarrollo y la selección de los modelos. Dado que la variable objetivo corresponde al retorno acumulado a 21 sesiones (forward_return_21d), las últimas 21 sesiones del período de precios disponible no pueden utilizarse como observaciones de desarrollo, ya que no existe información suficiente para calcular su retorno futuro completo. Por este motivo, aunque los precios disponibles alcanzan el 30 de diciembre de 2024, el último target válido del período de desarrollo se sitúa aproximadamente a principios de diciembre de 2024.

A partir de esta fecha se mantiene un período adicional de separación temporal, siguiendo el principio de las ventanas de purging y embargo empleado anteriormente en el esquema CPCV. Esta separación evita que exista solapamiento entre los horizontes de los retornos utilizados durante el desarrollo y las observaciones destinadas al test final. De este modo, el período Out-of-Sample comienza aproximadamente a mediados de enero de 2025.

### 3.1 Load OOS features

In [11]:
# =============================================================================
# Extended S&P 500 Price Dataset for OOS Testing
# =============================================================================

import pandas as pd

from src.data.download_data import (
    download_prices,
    remove_empty_tickers,
    validate_download,
)

# =============================================================================
# Configuration
# =============================================================================

START_DATE = "2010-01-01"

# OOS evaluation period ends on 15/07/2026.
# Additional data is required afterwards to calculate the 21-day forward return
# for the final OOS observations.
END_DATE = "2026-09-01"

INTERVAL = "1d"
AUTO_ADJUST = False

# =============================================================================
# S&P 500 Constituents
# =============================================================================

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

sp500 = pd.read_html(
    url,
    storage_options={"User-Agent": "Mozilla/5.0"},
)[0]

# Yahoo Finance uses "-" instead of "." in ticker symbols

tickers = (
    sp500["Symbol"]
    .str.replace(".", "-", regex=False)
    .tolist()
)

print(f"Requested tickers: {len(tickers)}")

# =============================================================================
# Download prices
# =============================================================================

prices = download_prices(
    tickers=tickers,
    START_DATE=START_DATE,
    END_DATE=END_DATE,
    INTERVAL=INTERVAL,
    AUTO_ADJUST=AUTO_ADJUST,
)

# =============================================================================
# Remove completely empty tickers
# =============================================================================

prices = remove_empty_tickers(
    prices
)

# =============================================================================
# Datetime and MultiIndex standardization
# =============================================================================

prices.index = pd.to_datetime(
    prices.index
)

prices.columns = pd.MultiIndex.from_tuples(
    [
        (str(c[0]), str(c[1]))
        for c in prices.columns
    ],
    names=["Price", "Ticker"],
)

# =============================================================================
# Validate downloaded data
# =============================================================================

validate_download(
    prices,
    tickers,
)

# =============================================================================
# Final dataset checks
# =============================================================================

print("\n" + "=" * 50)
print("EXTENDED PRICE DATASET")
print("=" * 50)

print(
    f"Start date : {prices.index.min().date()}"
)

print(
    f"End date   : {prices.index.max().date()}"
)

print(
    f"Rows       : {len(prices):,}"
)

print(
    f"Tickers    : "
    f"{prices.columns.get_level_values('Ticker').nunique():,}"
)

# =============================================================================
# Save extended dataset
# =============================================================================

prices.to_parquet(
    "../data/raw/sp500_prices_extended.parquet"
)

print(
    "\nSaved to:"
    "\n../data/raw/sp500_prices_extended.parquet"
)

Requested tickers: 503


[*********************100%***********************]  503 of 503 completed


DOWNLOAD SUMMARY
Requested tickers : 503
Valid downloads   : 503
Missing tickers   : 0
Empty tickers (NaN): 0

All tickers downloaded and validated successfully.

EXTENDED PRICE DATASET
Start date : 2010-01-04
End date   : 2026-08-10
Rows       : 4,175
Tickers    : 503

Saved to:
../data/raw/sp500_prices_extended.parquet


In [15]:
from src.preprocessing.factor_preprocessing import (
    winsorize_cross_sectional,
    rank_cross_sectional,
)

from src.features.liquidity import (
    compute_amihud_illiquidity,
)

from src.features.momentum import (
    compute_12_1_momentum,
)

from src.features.volatility import (
    compute_upside_volatility,
)

# =============================================================================
# Load extended price data
# =============================================================================

prices = pd.read_parquet(
    "../data/raw/sp500_prices_extended.parquet"
)

# =============================================================================
# OOS Evaluation Period
# =============================================================================

OOS_START = pd.Timestamp("2025-01-15")
OOS_END = pd.Timestamp("2026-07-15")

# =============================================================================
# Remove excluded securities
# =============================================================================

tickers_to_remove = [
    "SW",
    "AMCR",
]

prices = prices.drop(
    columns=tickers_to_remove,
    level=1,
    errors="ignore",
)

# =============================================================================
# Verify price data coverage
# =============================================================================

print("=" * 80)
print("PRICE DATA COVERAGE")
print("=" * 80)

print(
    f"\nAvailable price period: "
    f"{prices.index.min().date()} → {prices.index.max().date()}"
)

print(
    f"OOS evaluation period: "
    f"{OOS_START.date()} → {OOS_END.date()}"
)

# We need additional observations after OOS_END to calculate
# the 21-trading-day forward return for the final OOS observations.

assert prices.index.max() > OOS_END, (
    "Price data does not extend beyond OOS_END. "
    "Additional observations are required to compute the "
    "21-day forward return."
)

# =============================================================================
# Compute returns
# =============================================================================

adj_close = prices["Adj Close"]

simple_returns = adj_close.pct_change()

log_returns = np.log(
    adj_close / adj_close.shift(1)
)

# =============================================================================
# Compute selected factors
# =============================================================================

# -----------------------------------------------------------------------------
# 12-1 Momentum
# -----------------------------------------------------------------------------

momentum_12_1 = compute_12_1_momentum(
    prices
)

# -----------------------------------------------------------------------------
# Upside Volatility
# -----------------------------------------------------------------------------

upside_volatility = compute_upside_volatility(
    log_returns
)

# -----------------------------------------------------------------------------
# Amihud Illiquidity
# -----------------------------------------------------------------------------

amihud_illiquidity = compute_amihud_illiquidity(
    prices=prices,
    simple_returns=simple_returns,
)

log10_amihud = np.log10(
    amihud_illiquidity
)

# =============================================================================
# Combine factor series into a single panel DataFrame
# =============================================================================

factor_matrix = (
    pd.concat(
        {
            "momentum_12_1": momentum_12_1.stack(),
            "upside_volatility": upside_volatility.stack(),
            "log10_amihud": log10_amihud.stack(),
        },
        axis=1,
    )
    .rename_axis(["date", "ticker"])
    .reset_index()
)

# =============================================================================
# Cross-sectional winsorization
# =============================================================================

factor_cols = [
    "momentum_12_1",
    "upside_volatility",
    "log10_amihud",
]

df_win = winsorize_cross_sectional(
    factor_matrix,
    cols=factor_cols,
    p_low=0.01,
    p_high=0.99,
)

# =============================================================================
# Cross-sectional percentile rank
# =============================================================================

win_cols = [
    f"{col}_win"
    for col in factor_cols
]

df_rank = rank_cross_sectional(
    df_win,
    cols=win_cols,
)

# =============================================================================
# Build OOS feature matrix
# =============================================================================

X_oos = (
    df_rank[
        [
            "date",
            "ticker",
            "momentum_12_1_win_rank",
            "upside_volatility_win_rank",
            "log10_amihud_win_rank",
        ]
    ]
    .set_index(["date", "ticker"])
)

# =============================================================================
# Restrict to OOS evaluation period
# =============================================================================

oos_dates = X_oos.index.get_level_values("date")

X_oos = X_oos.loc[
    (oos_dates >= OOS_START)
    & (oos_dates <= OOS_END)
]

# =============================================================================
# Data integrity checks
# =============================================================================

print("\n" + "=" * 80)
print("OOS FEATURE DATASET")
print("=" * 80)

oos_dates = X_oos.index.get_level_values("date")

print(
    "\nOOS period:",
    oos_dates.min(),
    "→",
    oos_dates.max(),
)

print(
    f"Observations: {len(X_oos):,}"
)

print("\nFeatures:")
print(
    X_oos.columns.tolist()
)

print("\nMissing values:")
print(
    X_oos.isna().sum()
)

print("\nFeature ranges:")
print(
    X_oos.describe().loc[
        ["min", "max"]
    ]
)

# =============================================================================
# Remove observations with incomplete feature information
# =============================================================================

X_oos = X_oos.dropna()

print(
    f"\nObservations after removing missing features: "
    f"{len(X_oos):,}"
)

# =============================================================================
# Final assertions
# =============================================================================

assert len(X_oos) > 0, (
    "OOS feature dataset is empty. "
    "Check OOS_START, OOS_END, and price data coverage."
)

assert not X_oos.isna().any().any(), (
    "OOS feature dataset contains missing values."
)

assert oos_dates.min() >= OOS_START
assert oos_dates.max() <= OOS_END

print("\n" + "=" * 80)
print("OOS FEATURE DATASET READY")
print("=" * 80)

PRICE DATA COVERAGE

Available price period: 2010-01-04 → 2026-08-10
OOS evaluation period: 2025-01-15 → 2026-07-15

OOS FEATURE DATASET

OOS period: 2025-01-15 00:00:00 → 2026-07-15 00:00:00
Observations: 187,875

Features:
['momentum_12_1_win_rank', 'upside_volatility_win_rank', 'log10_amihud_win_rank']

Missing values:
momentum_12_1_win_rank        1499
upside_volatility_win_rank    1055
log10_amihud_win_rank          971
dtype: int64

Feature ranges:
     momentum_12_1_win_rank  upside_volatility_win_rank  log10_amihud_win_rank
min               -0.493976                   -0.493988                 -0.494
max                0.495984                    0.495992                  0.496

Observations after removing missing features: 186,376

OOS FEATURE DATASET READY


In [14]:
print("\nMissing percentage:")
print(
    X_oos.isna().mean() * 100
)


Missing percentage:
momentum_12_1_win_rank        0.797871
upside_volatility_win_rank    0.561544
log10_amihud_win_rank         0.516833
dtype: float64


   ### 3.2 Load OOS forward returns
   ### 3.3 Verify OOS period
   ### 3.4 Data integrity checks

## 4. Generate OOS Predictions
   ### 4.1 Generate predictions
   ### 4.2 Store predictions
   ### 4.3 Cross-sectional prediction analysis

## 5. OOS Predictive Performance
   ### 5.1 RMSE / MAE
   ### 5.2 Information Coefficient (IC) & Rank IC
   ### 5.3 IC Stability (IC Sharpe Ratio)
   ### 5.4 Hit Rate / Directional Accuracy
   ### 5.5 Temporal evolution of predictive power

## 6. OOS Diagnostics
   ### 6.1 Prediction distribution
   ### 6.2 Predicted vs. realized returns
   ### 6.3 Cross-sectional quantile/decile analysis (Monotonicity check)
   ### 6.4 Factor/model stability

## 7. Export OOS Results
   ### 7.1 OOS predictions
   ### 7.2 OOS metrics
   ### 7.3 OOS metadata

## 8. Conclusions